In [19]:
# 1. Bibliotecas padrão do Python
import warnings

# 2. Bibliotecas de terceiros
import pandas as pd
import plotly.express as px

# 3. Configurações globais
warnings.filterwarnings('ignore', category=FutureWarning)


In [20]:
df = pd.read_csv('/Users/renangomes/Desktop/political-ads-dashboard/data/Global_Cybersecurity_Threats_2015-2024.csv', encoding = 'UTF-8').copy()

df.head()

,Country,Year,Attack Type,Target Industry,Financial Loss (in Million $),Number of Affected Users,Attack Source,Security Vulnerability Type,Defense Mechanism Used,Incident Resolution Time (in Hours)
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,UK,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Country                              3000 non-null   object 
 1   Year                                 3000 non-null   int64  
 2   Attack Type                          3000 non-null   object 
 3   Target Industry                      3000 non-null   object 
 4   Financial Loss (in Million $)        3000 non-null   float64
 5   Number of Affected Users             3000 non-null   int64  
 6   Attack Source                        3000 non-null   object 
 7   Security Vulnerability Type          3000 non-null   object 
 8   Defense Mechanism Used               3000 non-null   object 
 9   Incident Resolution Time (in Hours)  3000 non-null   int64  
dtypes: float64(1), int64(3), object(6)
memory usage: 234.5+ KB


In [22]:
df.isna().sum()

Country                                0
Year                                   0
Attack Type                            0
Target Industry                        0
Financial Loss (in Million $)          0
Number of Affected Users               0
Attack Source                          0
Security Vulnerability Type            0
Defense Mechanism Used                 0
Incident Resolution Time (in Hours)    0
dtype: int64

In [23]:
df.describe().round(2)

,Year,Financial Loss (in Million $),Number of Affected Users,Incident Resolution Time (in Hours)
count,3000.00,3000.00,3000.00,3000.00
mean,2019.57,50.49,504684.14,36.48
std,2.86,28.79,289944.08,20.57
min,2015.00,0.50,424.00,1.00
25%,2017.00,25.76,255805.25,19.00
50%,2020.00,50.80,504513.00,37.00
75%,2022.00,75.63,758088.50,55.00
max,2024.00,99.99,999635.00,72.00


In [24]:
# Célula 4 — Renomear colunas para snake_case
df = df.rename(columns={
    'Country': 'country',
    'Year': 'year',
    'Attack Type': 'attack_type',
    'Target Industry': 'target_industry',
    'Financial Loss (in Million $)': 'financial_loss',
    'Number of Affected Users': 'affected_users',
    'Attack Source': 'attack_source',
    'Security Vulnerability Type': 'vulnerability_type',
    'Defense Mechanism Used': 'defense_mechanism',
    'Incident Resolution Time (in Hours)': 'resolution_time_hours'
})

In [25]:
#Conversão para category
cols = ['country', 'attack_type', 'target_industry', 'attack_source', 'vulnerability_type', 'defense_mechanism']

for col in cols:
    df.loc[:, col] = df[col].astype('category')

print(df.dtypes)

country                   object
year                       int64
attack_type               object
target_industry           object
financial_loss           float64
affected_users             int64
attack_source             object
vulnerability_type        object
defense_mechanism         object
resolution_time_hours      int64
dtype: object


In [26]:
# Criação da coluna sevetity com os quartis da coluna financial loss
df.loc[:, 'severity'] = pd.qcut(
    df['financial_loss'],
    q = 3,
    labels = ['baixo', 'médio', 'alto']
)

print(df['severity'].value_counts())

severity
baixo    1000
médio    1000
alto     1000
Name: count, dtype: int64


In [27]:
df.loc[:, 'response_speed'] = pd.cut(df['resolution_time_hours'], bins = [0,24,48,72], labels = ['rápido', 'médio','lento'])

In [28]:
df.describe(include = 'all')

,country,year,attack_type,target_industry,financial_loss,affected_users,attack_source,vulnerability_type,defense_mechanism,resolution_time_hours,severity,response_speed
count,3000,3000.000000,3000,3000,3000.000000,3000.000000,3000,3000,3000,3000.000000,3000,3000
unique,10,NaN,6,7,NaN,NaN,4,4,5,NaN,3,3
top,UK,NaN,DDoS,IT,NaN,NaN,Nation-state,Zero-day,Antivirus,NaN,baixo,médio
freq,321,NaN,531,478,NaN,NaN,794,785,628,NaN,1000,1026
mean,NaN,2019.570333,NaN,NaN,50.492970,504684.136333,NaN,NaN,NaN,36.476000,NaN,NaN
std,NaN,2.857932,NaN,NaN,28.791415,289944.084972,NaN,NaN,NaN,20.570768,NaN,NaN
min,NaN,2015.000000,NaN,NaN,0.500000,424.000000,NaN,NaN,NaN,1.000000,NaN,NaN
25%,NaN,2017.000000,NaN,NaN,25.757500,255805.250000,NaN,NaN,NaN,19.000000,NaN,NaN
50%,NaN,2020.000000,NaN,NaN,50.795000,504513.000000,NaN,NaN,NaN,37.000000,NaN,NaN
75%,NaN,2022.000000,NaN,NaN,75.630000,758088.500000,NaN,NaN,NaN,55.000000,NaN,NaN


País mais atacado: UK (321x)
Tipo de ataque mais usado: DDoS (531x)
Industria mais atacada: IT (478)
Mecanismo de defesa mais usado: Antivírus



In [29]:
# Ataques por ano
ataques_por_ano = df.groupby('year').size().reset_index(name = 'total_ataques')

print(ataques_por_ano)


   year  total_ataques
0  2015            277
1  2016            285
2  2017            319
3  2018            310
4  2019            263
5  2020            315
6  2021            299
7  2022            318
8  2023            315
9  2024            299


In [30]:
prejuizo_pais = df.groupby('country')['financial_loss'].agg(['mean', 'sum', 'count']).round(2)

prejuizo_pais.columns = ['média', 'total', 'quantidade_ataques']

prejuizo_pais = prejuizo_pais.sort_values('total', ascending = False)

print(prejuizo_pais)

           média     total  quantidade_ataques
country                                       
UK         51.41  16502.99                 321
Germany    54.27  15793.24                 291
Brazil     50.91  15782.62                 310
Australia  51.86  15403.00                 297
Japan      49.83  15197.34                 305
France     49.09  14972.28                 305
USA        51.61  14812.12                 287
Russia     49.95  14734.73                 295
India      47.29  14566.12                 308
China      48.81  13714.47                 281


In [31]:
#País com maior prejuízo: UK
#País com maior média de ataques: Germany

In [32]:
cross = pd.crosstab(df['attack_type'], df['target_industry'])

print(cross)

target_industry    Banking  Education  Government  Healthcare  IT  Retail  \
attack_type                                                                 
DDoS                    71         73          71          78  91      62   
Malware                 61         70          64          81  67      68   
Man-in-the-Middle       77         65          53          58  80      70   
Phishing                96         73          68          63  89      89   
Ransomware              69         71          72          77  74      71   
SQL Injection           71         67          75          72  77      63   

target_industry    Telecommunications  
attack_type                            
DDoS                               85  
Malware                            74  
Man-in-the-Middle                  56  
Phishing                           51  
Ransomware                         59  
SQL Injection                      78  


In [33]:
defesa_efic = df.groupby('defense_mechanism')['resolution_time_hours'].mean().sort_values()

print(defesa_efic)

defense_mechanism
Firewall              35.714530
Antivirus             36.573248
Encryption            36.589527
AI-based Detection    36.612350
VPN                   36.864379
Name: resolution_time_hours, dtype: float64


In [37]:
fig1 = px.line(
    ataques_por_ano,
    x='year',
    y='total_ataques',
    title='Evolução de Ataques Cibernéticos (2015–2024)',
    markers=True
)
fig1.show()


In [38]:
# Recalcular (corrigindo o bug de antes)
prejuizo_pais = (
    df.groupby('country')['financial_loss']
    .agg(['mean', 'sum', 'count'])
    .reset_index()
    .rename(columns={'mean': 'media', 'sum': 'total', 'count': 'qtd_ataques'})
    .sort_values('total', ascending=False)
)

fig2 = px.bar(
    prejuizo_pais,
    x='country',
    y='total',
    title='Prejuízo Total por País (em milhões USD)',
    color='total',
    color_continuous_scale='Reds'
)
fig2.show()


In [34]:
# Célula — Heatmap ataque x indústria
cross = pd.crosstab(df['attack_type'], df['target_industry'])

fig3 = px.imshow(
    cross,
    title='Frequência de Ataques por Tipo e Indústria',
    color_continuous_scale='Blues',
    text_auto=True,        # mostra o número dentro de cada célula
    aspect='auto'
)
fig3.show()


In [35]:
# Célula — Defesa x tempo médio de resolução
defesa_efic = (
    df.groupby('defense_mechanism')['resolution_time_hours']
    .mean()
    .reset_index()
    .sort_values('resolution_time_hours')
)

fig4 = px.bar(
    defesa_efic,
    x='resolution_time_hours',
    y='defense_mechanism',
    orientation='h',           # barras horizontais — melhor para labels longos
    title='Tempo Médio de Resolução por Mecanismo de Defesa (horas)',
    color='resolution_time_hours',
    color_continuous_scale='RdYlGn_r'   # verde = rápido, vermelho = lento
)
fig4.show()


In [39]:
fig5 = px.histogram(
    df,
    x='attack_type',
    color='severity',
    barmode='group',
    title='Distribuição de Severidade por Tipo de Ataque',
    category_orders={'severity': ['baixo', 'médio', 'alto']}
)
fig5.show()
